# Chapter 8. Deployment in Kubernetes

## Objective: Deploy versioning and production pipelines in K8s.

### We'll be using these Step-by-Step Guide:
### 1. Create K8s Manifests
### 2. Apply: kubectl apply -f deployment.yaml.
### 3. Auto-Deployment: In Airflow DAG, use KubernetesOperator to update deployment image tag with new version (e.g., iris-serving: v{mlflow_run_id}) if better.
### 4. Separate Namespaces: Training in 'training-ns', production in 'prod-ns'.
### 5. Retraining: If not better, trigger new DAG run.
### 6. Hands-On: Deploy to Minikube, access via minikube service iris-serving.


# 1. Create K8s Manifests
A manifest in Kubernetes is simply a YAML (or JSON) configuration file that describes the desired state of a resource in the cluster.

```
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-serving
spec:
  replicas: 1
  template:
    spec:
      containers:
      - name: serving
        image: iris-serving:latest
        ports:
        - containerPort: 5000
```

**Explaination:**

Deployment: Tells Kubernetes to run and manage one or more copies (replicas) of your container.

replicas: 1 → only one pod will be running.

containers → image: iris-serving:latest → runs your Docker image (iris-serving) built for serving predictions.

containerPort: 5000 → your app listens on port 5000 inside the pod.

## 2. Apply: `kubectl apply -f deployment.yaml.`

- kubectl is the CLI tool to interact with the Kubernetes cluster.

- apply → tells Kubernetes to create or update resources so that the cluster’s actual state matches the desired state described in the YAML manifest.

- -f deployment.yaml → means we are giving Kubernetes a file that contains resource definitions (like Deployments, Services, ConfigMaps).


## 3. Auto-Deployment:
In Airflow DAG, use KubernetesOperator to update deployment image tag with new version (e.g., iris-serving:v{mlflow_run_id}) if better.

**Theoretical Explanation:**

After training a new model, we need to deploy it only if it performs better.

In MLOps, this is automated by connecting the ML pipeline (training + evaluation) with the deployment pipeline (Kubernetes).

**Airflow acts as the orchestrator:**

- It checks model performance.

- If the model is better, it automatically updates the Kubernetes Deployment with the new image version.

- The Deployment controller then rolls out the updated Pods with the new container image.

The KubernetesOperator in Airflow provides a way to run Kubernetes tasks directly from DAGs. Here, it will be used to patch the Deployment and set a new Docker image tag (e.g., iris-serving:v1234 where 1234 is the mlflow_run_id).

This creates a **closed loop:**

**1. Train → Evaluate → Decide.**

**2. If better, Airflow updates Deployment → New model goes live.**

**3. If not better, pipeline either retrains or stops.**



## 4. Separate Namespaces: Training in 'training-ns', production in 'prod-ns'.

**What are Namespaces?**

- A namespace in Kubernetes is a way to logically isolate and organize resources within the same cluster.

Think of a cluster as a big apartment building and namespaces as different flats.

- Each flat (namespace) has its own resources, tenants, and rules.

- But they all share the same building (cluster infrastructure).

**Why separate namespaces in MLOps?**

**Training vs Production isolation:**

- training-ns → used for model training, experimentation, retraining jobs, and evaluation pipelines.

- prod-ns → used for stable, serving-ready deployments accessible to users.

**YAML Example — Namespaces**

Save as namespaces.yaml:

```
apiVersion: v1
kind: Namespace
metadata:
  name: training-ns
---
apiVersion: v1
kind: Namespace
metadata:
  name: prod-ns
```

#### Practical Flow with Namespaces

**Training DAG → training-ns**

- Launches jobs/pods for model retraining.

- Stores artifacts in MLflow or object storage.

**Evaluation → Decision**

- Compares metrics of the new model with the current one.

**Deployment DAG → prod-ns**

- If better, Airflow updates the Deployment in prod-ns with a new image tag.

- Production service stays isolated and stable.



## 5. Retraining: If not better, trigger new DAG run.

**Theoretical Explanation**

- In an MLOps pipeline, not every trained model is better than the existing production model.

- “Better” is usually measured using evaluation metrics such as accuracy, F1-score, ROC-AUC, etc.

- If the new model does not meet the improvement threshold, you do not deploy it to production.

- Instead, you trigger a new retraining cycle:

1. Adjust hyperparameters

2. Increase training data

3. Apply new preprocessing or augmentation

This ensures production always runs the best available model while experiments continue in the training namespace.


### Practical Implementation in Airflow

#### **Workflow:**

**1. Training DAG runs in training-ns**

- Trains a new model and logs metrics in MLflow.

**2. Evaluation Task**

- Compares the latest model metrics to production metrics.

- Example logic:

```
if new_model_metric > prod_model_metric:
    trigger_deploy_dag(mlflow_run_id)
else:
    trigger_retrain_dag()
```

**3. Trigger Retraining DAG**

- Airflow can trigger another DAG run using TriggerDagRunOperator:

```
from airflow.operators.trigger_dagrun import TriggerDagRunOperator

retrain = TriggerDagRunOperator(
    task_id="retrain_model",
    trigger_dag_id="training_dag",
    conf={"reason": "model_not_better"}
)
```

4. Loop Until Improvement

- This creates a feedback loop: train → evaluate → deploy or retrain.

- Ensures production always has the best-performing, stable model.